### Checks

`Urban Ward_Panel` sheets.


1. All TV_CODES should show up at least once. Either one shape for a town or village (NaN Ward code) or multiple ward shapes with codes.
2. Which TV_Codes have we been given multiple shapes for (wards)? Should match or overdeliver on their promise.

Add column that says "Delivery State": 
- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd

gpd.options.io_engine = "pyogrio"

In [ ]:
from gridsample.utils import save_shapefiles

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data_panel"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data"
CLEANED_DATA_DIR = (
    DATA_DIR / "02. Intermediate Outputs" / "00. Merge and Quality Checks" / "v1"
)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 0. Load request excel

In [ ]:
excel_file = (
    RAW_DATA_DIR / "00. Boundary Requests" / "SamplingOutput_Summary_IFS & Panel.xlsx"
)

In [ ]:
panel_df = pd.read_excel(excel_file, sheet_name="Urban Ward_Panel")

In [ ]:
panel_df

In [ ]:
# sort and reset index
panel_df = panel_df.sort_values(
    by=["State", "District", "Subdistrict", "TownVillage", "UrbanWardVillage"]
).reset_index(drop=True)

In [ ]:
panel_df

In [ ]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards.csv", index=False)

In [ ]:
# get unique across district and subdistrict both
len(panel_df[["State", "District", "Subdistrict"]].drop_duplicates())

#### Check for duplicated wards in our own sample

In [ ]:
duplicated_sampled_wards = panel_df[
    panel_df[["TownVillage", "UrbanWardVillage"]].duplicated(keep=False)
].sort_values(["TownVillage", "UrbanWardVillage"])
duplicated_sampled_wards

In [ ]:
duplicated_sampled_wards.to_csv(
    CLEANED_DATA_DIR / "Duplicated Sampled Wards.csv", index=False
)

## 1. Load all boundaries

In [ ]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]

In [ ]:
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True)

In [ ]:
gdf = gdf.drop_duplicates()

#### Fix Chittoor issue

In [ ]:
# gdf[gdf["Dist_N"] == "Chittoor"]

In [ ]:
gdf.loc[gdf["TV_C"] == 803014, ["SubDist_N", "SubDist_C"]] = ["Tirupati (Urban)", 5383]

In [ ]:
# gdf[gdf["Dist_N"] == "Chittoor"]

#### Fix Patna ward PCA_ID issue

In [ ]:
# gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3")]

In [ ]:
gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3"), "PCA_ID"] = "801373-3"

In [ ]:
# gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3")]

#### Replace some MDDS codes with TV and Ward codes as census expects

In [ ]:
# gdf.loc[(gdf["TV_C"].isin([519923, 574077]))]

In [ ]:
gdf.loc[gdf["TV_C"] == 519923, ["TV_C", "Ward_C", "PCA_ID"]] = [802596, 18, "802596-18"]
gdf.loc[gdf["TV_C"] == 574077, ["TV_C", "Ward_C", "PCA_ID"]] = [802918, 157, "802918-157"]

## 2. Checks

### Any MapSolve rows with missing TV code?

In [ ]:
gdf_no_TV_code_filtered = gdf[gdf["TV_C"].isna()]
gdf_no_TV_code_filtered

In [ ]:
gdf_no_TV_code_filtered.to_csv(
    CLEANED_DATA_DIR / "MapSolve Data without TV Codes.csv", index=False
)

### Request satisfaction check

In [ ]:
given_states_list = list(gdf["State_C"].unique())
given_states_list.append(
    90
)  # manually add 90 for Telangana / Andhra Pradesh discrepency
len(given_states_list)

In [ ]:
panel_df.loc[panel_df["State"].isin(given_states_list), "State_Name"].unique()

In [ ]:
panel_df["State Shared by MapSolve"] = False
panel_df.loc[
    panel_df["State"].isin(given_states_list), "State Shared by MapSolve"
] = True

In [ ]:
len(panel_df[~panel_df["State Shared by MapSolve"]])

In [ ]:
panel_df["State Changed"] = "No"
panel_df.loc[panel_df["State_Name"] == "TELANGANA", "State Changed"] = (
    "Previously Andhra Pradesh"
)

In [ ]:
panel_df

#### Check for wards

In [ ]:
panel_df["PCA_ID"] = (
    panel_df["TownVillage"].astype(str)
    + "-"
    + panel_df["UrbanWardVillage"].astype(str)
)
given_ward_ids = gdf["PCA_ID"].unique()

panel_df["Ward Boundary Given"] = False
panel_df.loc[panel_df["PCA_ID"].isin(given_ward_ids), "Ward Boundary Given"] = True

In [ ]:
panel_df["PCA_ID"]

In [ ]:
len(set(panel_df["PCA_ID"]).intersection(given_ward_ids))

#### Check for TownVillage codes

In [ ]:
given_TV_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].notna(),
        "TV_C",
    ].astype(int)
)

panel_df["TV Boundary Given"] = False
panel_df.loc[
    panel_df["TownVillage"].astype(int).isin(given_TV_ids),
    "TV Boundary Given",
] = True

#### Check for SubDistricts

In [ ]:
given_subdist_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].isna() & gdf["SubDist_C"].notna(),
        "SubDist_C",
    ].astype(int)
)

panel_df["SubDistrict Boundary Given"] = False
panel_df.loc[
    panel_df["Subdistrict"].astype(int).isin(given_subdist_ids),
    "SubDistrict Boundary Given",
] = True

#### Fill in Delivery State

- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

In [ ]:
## baseline
panel_df["Delivery State"] = "BAD - No boundary(s) given"

## subdistrict
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "BAD - Subdistrict boundary given but Ward boundaries expected"

# tripura > dukli special case
panel_df.loc[
    (panel_df["State_Name"] == "TRIPURA") & (panel_df["Subd_Name"] == "Dukli"),
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"

## town/village
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "GOOD - Town/Village boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "BAD - Town/Village boundary given but Ward boundaries expected"

## ward
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "BETTER - Ward boundary given but only TV or Subdistrict was expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "GOOD - Ward boundary given as expected"

In [ ]:
panel_df["Delivery State"].value_counts()

#### Add `PSU Type` column

In [ ]:
psu_mapping = {
    "BETTER - Ward boundary given but only TV or Subdistrict was expected": "ward",
    "GOOD - Town/Village boundary given as expected": "town_village",
    "GOOD - Ward boundary given as expected": "ward",
    "GOOD - Subdistrict boundary given as expected": "subdistrict",
    "BAD - Town/Village boundary given but Ward boundaries expected": "town_village",
    "BAD - No boundary(s) given": "none",
    "BAD - Subdistrict boundary given but Ward boundaries expected": "subdistrict",
}
# create a new column in panel_df called PSU Type
panel_df["PSU Type"] = panel_df["Delivery State"].map(psu_mapping)

#### Add `PSU ID` column (unique overall ID)

In [ ]:
# add an ID column that is unique across all rows. It should be WARD_{PCA_ID} if the PSU Type is "ward", TV_{TV Code} if the PSU Type is "town_village", and SUBDISTRICT_{Subdistrict Code} if the PSU Type is "subdistrict"
panel_df["PSU ID"] = panel_df.apply(
    lambda row: f"WARD_{row['PCA_ID']}"
    if row["PSU Type"] == "ward"
    else f"TV_{int(row['TownVillage'])}"
    if row["PSU Type"] == "town_village"
    else f"SUBDISTRICT_{int(row['Subdistrict'])}",
    axis=1,
).astype(str)

#### Add `Ward Count` column

In [ ]:
panel_df["Ward Count"] = 0

panel_df.loc[panel_df["PSU Type"] == "ward", "Ward Count"] = 1
panel_df.loc[panel_df["PSU Type"] == "town_village", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "town_village"]
    .groupby("TownVillage")["UrbanWardVillage"]
    .transform("count")
)
panel_df.loc[panel_df["PSU Type"] == "subdistrict", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "subdistrict"]
    .groupby("Subdistrict")["UrbanWardVillage"]
    .transform("count")
)
panel_df["Ward Count"] = panel_df["Ward Count"].astype(int)

#### Reorder columns

In [ ]:
reordered_columns = [
    "State",
    "State_Name",
    "District",
    "District_Name",
    "Subdistrict",
    "Subd_Name",
    "TownVillage",
    "UrbanWardVillage",
    "WardVillage_Name",
    "PCA_ID",
    "TRU",
    "WardVillage_Pop",
    "Subd_Pop",
    "State_Pop",
    "WardVillageID",
    "Ward Boundary Available with MapSolve",
    "Included in Panel",
    "State Shared by MapSolve",
    "State Changed",
    "Ward Boundary Given",
    "TV Boundary Given",
    "SubDistrict Boundary Given",
    "Delivery State",
    "PSU Type",
    "PSU ID",
    "Ward Count",
]
panel_df = panel_df[reordered_columns]

#### Save

In [ ]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards with Quality Checks.csv", index=False)

In [ ]:
if not panel_df["State Shared by MapSolve"].all():
    print("There are states in the merged data that are not shared by MapSolve:")
    print(panel_df[~panel_df["State Shared by MapSolve"]]["State"].unique())
    print("Saving the states for which we DO have data separately:")
    panel_df[panel_df["State Shared by MapSolve"]].to_csv(
        CLEANED_DATA_DIR / "Panel Wards with Quality Checks - Shared States.csv",
        index=False,
    )

#### Save per-state stats

In [ ]:
# Get counts of delivery states by state
delivery_state_counts = (
    panel_df[panel_df["State"].isin(given_states_list)]
    .groupby("State")["Delivery State"]
    .value_counts()
)

# Convert to a more readable DataFrame format
delivery_state_pivot = delivery_state_counts.unstack(fill_value=0).reset_index()

# Sort by state code
delivery_state_pivot = delivery_state_pivot.sort_values(by="State")

# Add state names for better readability
state_name_mapping = (
    panel_df[["State", "State_Name"]]
    .drop_duplicates()
    .set_index("State")["State_Name"]
)
delivery_state_pivot["State_Name"] = delivery_state_pivot["State"].map(
    state_name_mapping
)

# Reorder columns to show State_Name first
columns = ["State", "State_Name"] + [
    col for col in delivery_state_pivot.columns if col not in ["State", "State_Name"]
]
delivery_state_pivot = delivery_state_pivot[columns]

# transform the DataFrame to have the delivery states as rows and state names as columns
delivery_state_pivot.drop(columns=["State"], inplace=True)
delivery_state_pivot = delivery_state_pivot.set_index("State_Name").T.reset_index()
delivery_state_pivot

In [ ]:
# add a total column as the second column
delivery_state_pivot.insert(1, "Total", delivery_state_pivot.iloc[:, 1:].sum(axis=1))

In [ ]:
delivery_state_pivot

In [ ]:
delivery_state_pivot.to_csv(
    CLEANED_DATA_DIR / "Delivery State Counts By State.csv", index=False
)

#### Old

In [ ]:
# unique_requested_tv_codes = set(
#     panel_df[panel_df["State Shared by MapSolve"]]["TownVillage"]
# )
# unique_received_tv_codes = set(gdf.loc[~gdf["TV_C"].isna(), "TV_C"].astype(int))
# matched_tv_codes = unique_received_tv_codes.intersection(unique_requested_tv_codes)

In [ ]:
# print("Number of requested TV codes:", len(unique_requested_tv_codes))
# print("Number of received TV codes:", len(unique_received_tv_codes))

# print("Number of matched TV codes:", len(matched_tv_codes))
# print(
#     "Number of TV codes not received:",
#     len(unique_requested_tv_codes.difference(matched_tv_codes)),
# )

In [ ]:
# # show the received breakdown by state
# received_pivot_table = (
#     panel_df[panel_df["State Shared by MapSolve"]]
#     .groupby(
#         [
#             "State",
#             "State_Name",
#             "District",
#             "District_Name",
#             "Subdistrict",
#             "Subd_Name",
#         ]
#     )["Delivery State"]
#     .value_counts()
#     .unstack(fill_value=0)
# ).reset_index()

# # received_pivot_table.rename(
# #     columns={True: "TV_Codes Received", False: "TV_Codes Not Received"}, inplace=True
# # )

In [ ]:
# received_pivot_table

In [ ]:
# pivot_table_not_received = received_pivot_table[
#     received_pivot_table["BAD - No boundary(s) given"] > 0
# ]
# pivot_table_not_received


In [ ]:
# pivot_table_not_received.to_csv(
#     CLEANED_DATA_DIR / "Requested TV Codes Not Received Subdistrict Breakdown.csv",
#     index=False,
# )

In [ ]:
# requested_tv_codes_not_received = panel_df[
#     (panel_df["State Shared by MapSolve"])
#     & (panel_df["Delivery State"] == "BAD - No boundary(s) given")
# ]
# requested_tv_codes_not_received

In [ ]:
# requested_tv_codes_not_received.to_csv(
#     CLEANED_DATA_DIR / "Requested TV Codes Not Received.csv", index=False
# )

## Post-sampling checks

In [ ]:
# check out district Chittoor, tv_code: 803014. Seems like we're missing the subdistrict name and ID is changed.

# ignore: (NCT of) Delhi, Tripura, Meghalaya

In [ ]:
# OUTPUT_DATA_DIR = DATA_DIR / "03. Final Outputs"

In [ ]:
# v4 = gpd.read_parquet(
#     OUTPUT_DATA_DIR
#     / "v4"
#     / "01. Sampled Rooftop Data"
#     / "sampled_rooftops_snapped_points.parquet"
# )
# v4["source_version"] = "v4"

# v6_uttarakhand = gpd.read_parquet(
#     OUTPUT_DATA_DIR
#     / "v6_uttarakhand"
#     / "01. Sampled Rooftop Data"
#     / "Uttarakhand"
#     / "Uttarakhand_sampled_rooftops_snapped_points.parquet"
# )
# v6_uttarakhand["source_version"] = "v6_uttarakhand"

# v7_leftouts = gpd.read_parquet(
#     OUTPUT_DATA_DIR
#     / "v7_leftouts"
#     / "01. Sampled Rooftop Data"
#     / "sampled_rooftops_snapped_points.parquet"
# )
# v7_leftouts["source_version"] = "v7_leftouts"

In [ ]:
# combined_rooftops_gdf = gpd.GeoDataFrame(
#     pd.concat([v4, v6_uttarakhand, v7_leftouts], ignore_index=True)
# )

In [ ]:
# combined_rooftops_gdf.columns

In [ ]:
# duplicated_gdf = combined_rooftops_gdf[
#     combined_rooftops_gdf["Rooftop Unique ID"].duplicated(keep=False)
# ].sort_values("Rooftop Unique ID")

In [ ]:
# # load Red's DQ checks
# missing_ward_df = pd.read_excel(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_v4.xlsx",
#     sheet_name="ward_unmerged",
# )
# missing_tv_df = pd.read_excel(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_v4.xlsx",
#     sheet_name="tv_unmerged",
# )
# missing_subd_df = pd.read_excel(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_v4.xlsx",
#     sheet_name="subdistrict_unmerged",
# )

In [ ]:
# missing_ward_df["psutype"] = missing_ward_df["deliverystate"].map(psu_mapping)
# missing_tv_df["psutype"] = missing_tv_df["deliverystate"].map(psu_mapping)
# missing_subd_df["psutype"] = missing_subd_df["deliverystate"].map(psu_mapping)

### Ward

In [ ]:
# missing_ward_df.groupby("deliverystate").size()

In [ ]:
# # add a new column that says "PSU Type in Rooftop Sample" that is true if found, False if not
# missing_ward_df["PSU Type in Rooftop Sample"] = False

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "ward",
#     "PSU Type in Rooftop Sample",
# ] = missing_ward_df["pca_id"].isin(combined_rooftops_gdf["PCA_ID"].unique())

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "town_village",
#     "PSU Type in Rooftop Sample",
# ] = missing_ward_df["townvillage"].isin(
#     combined_rooftops_gdf[
#         combined_rooftops_gdf["Ward Code"].isnull()
#         & combined_rooftops_gdf["TV Code"].notna()
#     ]["TV Code"].unique()
# )

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "subdistrict",
#     "PSU Type in Rooftop Sample",
# ] = missing_ward_df["subdistrict"].isin(
#     combined_rooftops_gdf[
#         (combined_rooftops_gdf["Ward Code"].isnull())
#         & (combined_rooftops_gdf["TV Code"].isnull())
#         & (combined_rooftops_gdf["Subdistrict Code"].notna())
#     ]["Subdistrict Code"].unique()
# )

# # add a new column that says "PSU Type in MapSolve Shapes" that is based on the "gdf" dataset that is true if found, False if not
# missing_ward_df["PSU Type in MapSolve Shapes"] = False

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "ward",
#     "PSU Type in MapSolve Shapes",
# ] = missing_ward_df["pca_id"].isin(gdf["PCA_ID"].unique())

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "town_village",
#     "PSU Type in MapSolve Shapes",
# ] = missing_ward_df["townvillage"].isin(
#     gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()]["TV_C"].unique()
# )

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "subdistrict",
#     "PSU Type in MapSolve Shapes",
# ] = missing_ward_df["subdistrict"].isin(
#     gdf[(gdf["Ward_C"].isnull()) & (gdf["TV_C"].isnull()) & (gdf["SubDist_C"].notna())][
#         "SubDist_C"
#     ].unique()
# )

In [ ]:
# missing_ward_df

In [ ]:
# missing_ward_df["PSU Type in Rooftop Sample"].value_counts()

### TV

In [ ]:
# missing_tv_df.groupby("deliverystate").size()

In [ ]:
# # add a new column that says "PSU Type in Rooftop Sample" that is true if found, False if not
# missing_tv_df["PSU Type in Rooftop Sample"] = False

# missing_tv_df.loc[
#     missing_tv_df["psutype"].isin(["ward"]),
#     "PSU Type in Rooftop Sample",
# ] = missing_tv_df["pca_id"].isin(combined_rooftops_gdf["PCA_ID"].unique())

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "town_village",
#     "PSU Type in Rooftop Sample",
# ] = missing_tv_df["tv_id"].isin(
#     combined_rooftops_gdf[
#         combined_rooftops_gdf["Ward Code"].isnull()
#         & combined_rooftops_gdf["TV Code"].notna()
#     ]["TV Code"].unique()
# )

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "subdistrict",
#     "PSU Type in Rooftop Sample",
# ] = missing_tv_df["subdistrict"].isin(
#     combined_rooftops_gdf[
#         (combined_rooftops_gdf["Ward Code"].isnull())
#         & (combined_rooftops_gdf["TV Code"].isnull())
#         & (combined_rooftops_gdf["Subdistrict Code"].notna())
#     ]["Subdistrict Code"].unique()
# )

# # add a new column that says "PSU Type in MapSolve Shapes" that is based on the "gdf" dataset that is true if found, False if not
# missing_tv_df["PSU Type in MapSolve Shapes"] = False

# missing_tv_df.loc[
#     missing_tv_df["psutype"].isin(["ward"]),
#     "PSU Type in MapSolve Shapes",
# ] = missing_tv_df["pca_id"].isin(gdf["PCA_ID"].unique())

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "town_village",
#     "PSU Type in MapSolve Shapes",
# ] = missing_tv_df["tv_id"].isin(
#     gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()]["TV_C"].unique()
# )

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "subdistrict",
#     "PSU Type in MapSolve Shapes",
# ] = missing_tv_df["subdistrict"].isin(
#     gdf[(gdf["Ward_C"].isnull()) & (gdf["TV_C"].isnull()) & (gdf["SubDist_C"].notna())][
#         "SubDist_C"
#     ].unique()
# )

In [ ]:
# missing_tv_df

In [ ]:
# missing_tv_df["PSU Type in Rooftop Sample"].value_counts()

### Subdistrict

In [ ]:
# missing_subd_df.groupby("deliverystate").size()

In [ ]:
# # add a new column that says "PSU Type in Rooftop Sample" that is true if found, False if not
# missing_subd_df["PSU Type in Rooftop Sample"] = False

# missing_subd_df.loc[
#     missing_subd_df["psutype"].isin(["ward"]),
#     "PSU Type in Rooftop Sample",
# ] = missing_subd_df["pca_id"].isin(combined_rooftops_gdf["PCA_ID"].unique())

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "town_village",
#     "PSU Type in Rooftop Sample",
# ] = missing_subd_df["tv_id"].isin(
#     combined_rooftops_gdf[
#         combined_rooftops_gdf["Ward Code"].isnull()
#         & combined_rooftops_gdf["TV Code"].notna()
#     ]["TV Code"].unique()
# )

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "subdistrict",
#     "PSU Type in Rooftop Sample",
# ] = missing_subd_df["subdistrict_census_id"].isin(
#     combined_rooftops_gdf[
#         (combined_rooftops_gdf["Ward Code"].isnull())
#         & (combined_rooftops_gdf["TV Code"].isnull())
#         & (combined_rooftops_gdf["Subdistrict Code"].notna())
#     ]["Subdistrict Code"].unique()
# )

# # add a new column that says "PSU Type in MapSolve Shapes" that is based on the "gdf" dataset that is true if found, False if not
# missing_subd_df["PSU Type in MapSolve Shapes"] = False

# missing_subd_df.loc[
#     missing_subd_df["psutype"].isin(["ward"]),
#     "PSU Type in MapSolve Shapes",
# ] = missing_subd_df["pca_id"].isin(gdf["PCA_ID"].unique())

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "town_village",
#     "PSU Type in MapSolve Shapes",
# ] = missing_subd_df["townvillage"].isin(
#     gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()]["TV_C"].unique()
# )

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "subdistrict",
#     "PSU Type in MapSolve Shapes",
# ] = missing_subd_df["subdistrict_census_id"].isin(
#     gdf[(gdf["Ward_C"].isnull()) & (gdf["TV_C"].isnull()) & (gdf["SubDist_C"].notna())][
#         "SubDist_C"
#     ].unique()
# )

In [ ]:
# missing_subd_df["PSU Type in Rooftop Sample"].value_counts()

In [ ]:
# # recreate excel and save to file
# with pd.ExcelWriter(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_updated_v4.xlsx"
# ) as writer:
#     missing_ward_df.to_excel(writer, sheet_name="ward_unmerged", index=False)
#     missing_tv_df.to_excel(writer, sheet_name="tv_unmerged", index=False)
#     missing_subd_df.to_excel(writer, sheet_name="subdistrict_unmerged", index=False)


### Get the shapes that were missed

In [ ]:
# missing_tv_df.rename(columns={"tv_id": "TV Code"}, inplace=True)
# missing_ward_df.rename(columns={"townvillage": "TV Code"}, inplace=True)
# missing_subd_df.rename(columns={"TownVillage": "TV Code"}, inplace=True)

In [ ]:
# combined_missing_df = pd.concat(
#     [missing_ward_df, missing_tv_df, missing_subd_df],
#     ignore_index=True,
# )

In [ ]:
# combined_missing_df.drop(columns=["townvillage", "tv_id", "TownVillage"], inplace=True)

In [ ]:
# unavailable_shapes_df = combined_missing_df[~combined_missing_df["PSU Type in MapSolve Shapes"]]
# unavailable_shapes_df

In [ ]:
# unavailable_shapes_df.to_csv(
#     CLEANED_DATA_DIR / "Unavailable Shapes in MapSolve.csv",
#     index=False,
# )

In [ ]:
# # filter out the rows where "PSU Type in Rooftop Sample" is True
# combined_missing_df = combined_missing_df[
#     ~combined_missing_df["PSU Type in Rooftop Sample"] &
#     combined_missing_df["PSU Type in MapSolve Shapes"]
# ]
# combined_missing_df

In [ ]:
# # drop any rows in states DELHI, NCT OF DELHI, TRIPURA, MEGHALAYA
# combined_missing_df = combined_missing_df[
#     ~combined_missing_df["State_Name"].isin(
#         ["DELHI", "NCT OF DELHI", "TRIPURA", "MEGHALAYA"]
#     )
# ]
# combined_missing_df

In [ ]:
# # deduplicate to avoid merging issues
# ward_missing_df = combined_missing_df[
#     combined_missing_df["psutype"] == "ward"
# ].drop_duplicates(subset=["pca_id"])
# tv_missing_df = combined_missing_df[
#     combined_missing_df["psutype"] == "town_village"
# ].drop_duplicates(subset=["TV Code"])
# subd_missing_df = combined_missing_df[
#     combined_missing_df["psutype"] == "subdistrict"
# ].drop_duplicates(subset=["subd_name"])

In [ ]:
# combined_missing_df[combined_missing_df["deliverystate"] == "BAD - Subdistrict boundary given but Ward boundaries expected"].to_csv(
#     CLEANED_DATA_DIR / "Ward Expected but Subdistrict Given.csv",
#     index=False,
# )

In [ ]:
# pd.concat(
#     [ward_missing_df, tv_missing_df, subd_missing_df],
#     ignore_index=True,
# )

In [ ]:
# # Ward shapes
# leftout_wards_gdf = gdf.merge(
#     ward_missing_df,
#     left_on="PCA_ID",
#     right_on="pca_id",
#     how="inner",
#     suffixes=("", "_missing"),
# )

# # TV shapes
# leftout_tvs_gdf = gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()].merge(
#     tv_missing_df,
#     left_on="TV_C",
#     right_on="TV Code",
#     how="inner",
#     suffixes=("", "_missing"),
# )

# # Subdistrict shapes
# leftout_subd_gdf = gdf[
#     gdf["Ward_C"].isnull() & gdf["TV_C"].isnull() & gdf["SubDist_C"].notna()
# ].merge(
#     subd_missing_df,
#     left_on="SubDist_N",
#     right_on="subd_name",
#     how="inner",
#     suffixes=("", "_missing"),
# )

In [ ]:
# all_leftout_gdf_merge = pd.concat(
#     [leftout_wards_gdf, leftout_tvs_gdf, leftout_subd_gdf],
#     ignore_index=True,
# )

In [ ]:
# all_leftout_gdf_merge.duplicated(keep=False).sum()

In [ ]:
# all_leftout_gdf_merge

In [ ]:
# all_leftout_gdf_merge.plot()

In [ ]:
# all_leftout_gdf_merge = all_leftout_gdf_merge[
#     [
#         "UID",
#         "PCA_ID",
#         "State_C",
#         "State_N",
#         "Dist_C",
#         "Dist_N",
#         "SubDist_C",
#         "SubDist_N",
#         "TV_C",
#         "TV_N",
#         "Ward_C",
#         "TOT_P",
#         "geometry",
#         "state",
#         "state_name",
#         "district",
#         "district_name",
#         "subdistrict",
#         "subd_name",
#         # "townvillage",
#         "TV Code",
#         "ward_id",
#         "wardvillage_name",
#         "pca_id",
#         "tru",
#         "wardvillage_pop",
#         "subd_pop",
#         "state_pop",
#         "wardvillageid",
#         "wardboundaryavailablewithmapsolv",
#         "sourcesheet",
#         "sampledforpanel",
#         "sampledforpanelround2",
#         "sampledforifs",
#         "statesharedbymapsolve",
#         "statechanged",
#         "wardboundarygiven",
#         "tvboundarygiven",
#         "subdistrictboundarygiven",
#         "deliverystate",
#         "psutype",
#         "wardcountifs",
#         "wardcountpanel",
#         "wardcountpanelround2",
#         "wardcountall",
#         "state_census_id",
#         "state_census_name",
#         "district_census_id",
#         "district_census_name",
#         "subdistrict_census_id",
#         "subdistrict_census_name",
#         # "tv_id",
#         "tv_name",
#         "target_psu",
#         "target_psu_type",
#         "targets_count",
#         "target_pin_type",
#         "pin_id",
#         "pin_name",
#         "gadm_state_name",
#         "gadm_district_name",
#         "gadm_subdistrict_name",
#         "google_maps",
#         "latitude",
#         "longitude",
#         "target_id",
#         "my_maps",
#         "State",
#         "State_Name",
#         "District",
#         "District_Name",
#         "Subdistrict",
#         "Subd_Name",
#         # "TownVillage",
#         "WardVillage_Name",
#         "TRU",
#         "WardVillage_Pop",
#         "Subd_Pop",
#         "State_Pop",
#         "WardVillageID",
#         "WardBoundaryAvailablewithMap",
#         "_m2",
#         "_merge",
#         "PSU Type in Rooftop Sample",
#         "PSU Type in MapSolve Shapes",
#         "urbanwardvillage",
#         "UrbanWardVillage",
#         "tv_ward_count",
#         "tv_sample_count",
#         "subdist_ward_count",
#         "subdist_sample_count",
#         "_m3",
#     ]
# ]

In [ ]:
# all_leftout_gdf_merge["UID"] = all_leftout_gdf_merge["UID"].astype(float)

In [ ]:
# save_shapefiles(
#     all_leftout_gdf_merge,
#     CLEANED_DATA_DIR / "Leftout Shapes",
#     "leftout_shapes_after_v4_v6_uttarakhand_v7_leftouts",
#     formats=["csv", "parquet", "kml"],
# )